In [1]:
import os, sys
if os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")
sys.path.insert(0, os.getcwd())

from collections import Counter
import numpy as np
import pandas as pd
from src.classifier.data import CATEGORIES, clean_text
from src.classifier.split import load_split, apply_split

df = pd.read_csv("data/opp115_segments.csv")
df["text"] = df["text"].map(clean_text)
test = apply_split(df, load_split())["test"]
yte = test[CATEGORIES].values
pred = (np.load("results/test_probs_bert_v2.npy") > 0.5).astype(int)

rows = []
for i in range(len(test)):
    true = [CATEGORIES[j] for j in range(len(CATEGORIES)) if yte[i, j] == 1]
    got = [CATEGORIES[j] for j in range(len(CATEGORIES)) if pred[i, j] == 1]
    missed = [c for c in true if c not in got]
    extra = [c for c in got if c not in true]
    if missed or extra:
        rows.append({
            "policy": test.loc[i, "policy_file"], "segment_id": test.loc[i, "segment_id"],
            "n_words": len(test.loc[i, "text"].split()),
            "text": test.loc[i, "text"][:600],
            "true": "; ".join(true), "predicted": "; ".join(got),
            "missed": "; ".join(missed), "extra": "; ".join(extra), "error_type": "",
        })
errs = pd.DataFrame(rows)
print(len(errs), "of", len(test), "test segments have at least one error")
print("missed labels:", Counter(c for m in errs["missed"] for c in m.split("; ") if c))
print("extra labels: ", Counter(c for m in errs["extra"] for c in m.split("; ") if c))

219 of 663 test segments have at least one error
missed labels: Counter({'Other': 71, 'Third Party Sharing/Collection': 17, 'First Party Collection/Use': 14, 'User Choice/Control': 13, 'International and Specific Audiences': 8, 'Data Security': 6, 'Data Retention': 5, 'User Access, Edit and Deletion': 4, 'Policy Change': 4, 'Do Not Track': 2})
extra labels:  Counter({'First Party Collection/Use': 43, 'Other': 40, 'Third Party Sharing/Collection': 34, 'User Choice/Control': 32, 'International and Specific Audiences': 8, 'Policy Change': 7, 'User Access, Edit and Deletion': 6, 'Data Retention': 2, 'Data Security': 2})


In [2]:
errs.sample(40, random_state=42).to_csv(
    "results/error_analysis_sample.csv", index=False, encoding="utf-8-sig")
# Policy Change was the class where the baseline won
errs[errs["true"].str.contains("Policy Change") | errs["predicted"].str.contains("Policy Change")]

,policy,segment_id,n_words,text,true,predicted,missed,extra,error_type
9,1099_enthusiastnetwork.com,34,95,XIII. ACCEPTANCE OF THESE TERMS; CHANGES TO OU...,User Choice/Control,Policy Change; Other,User Choice/Control,Policy Change; Other,
28,1300_bankofamerica.com,1,5,"Last updated July 24, 2014",Other,Policy Change; Other,,Policy Change,
61,135_instagram.com,2,82,We're updating our Privacy Policy to highlight...,User Choice/Control,Policy Change; Other,User Choice/Control,Policy Change; Other,
62,135_instagram.com,3,26,Our new Privacy Policy is effective on January...,Other,Policy Change; Other,,Policy Change,
84,1618_sltrib.com,28,65,Opt-Out Exception: Even if the registered user...,Other,Policy Change,Other,Policy Change,
119,1683_dailynews.com,21,65,Opt-Out Exception: Even if the registered user...,Other,Policy Change,Other,Policy Change,
124,186_abcnews.com,2,7,"Privacy Policy Last Modified: June 19, 2015",Policy Change; Other,Other,Policy Change,,
135,186_abcnews.com,39,35,Notice Notice may be by email to you at the la...,,Policy Change,,Policy Change,
147,228_gawker.com,24,13,9. Last Update This Privacy Policy was most re...,Policy Change; Other,Other,Policy Change,,
157,453_barnesandnoble.com,3,14,To review the prior version of Barnes & Noble'...,Policy Change,Other,Policy Change,Other,


In [3]:
from transformers import AutoTokenizer

tok = AutoTokenizer.from_pretrained("dralsarrani/opp115-bert")
full = test.set_index(["policy_file", "segment_id"])["text"]
errs["n_tokens"] = [len(tok(full[(p, s)])["input_ids"])
                    for p, s in zip(errs["policy"], errs["segment_id"])]


def auto_tags(r):
    tags = []
    if r["true"] == "":
        tags.append("gold-empty (no label survived the vote)")
    if r["n_tokens"] > 256:
        tags.append("truncated (over 256 tokens)")
    if r["n_words"] < 12:
        tags.append("very short (under 12 words)")
    if "Other" in r["missed"] or "Other" in r["extra"]:
        tags.append("Other boundary")
    if r["missed"] and ";" in r["true"]:
        tags.append("multi-label gold, model missed one")
    if r["missed"] and r["extra"]:
        tags.append("swap (missed one, added another)")
    elif r["missed"]:
        tags.append("missed only")
    else:
        tags.append("extra only")
    return tags


errs["tags"] = errs.apply(auto_tags, axis=1)
counts = errs["tags"].explode().value_counts().to_frame("errors")
counts["share of 219"] = (counts["errors"] / len(errs)).round(2)
counts

c:\Users\danah\miniconda3\envs\aiep\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


,errors,share of 219
tags,,
Other boundary,111,0.51
extra only,81,0.37
"swap (missed one, added another)",79,0.36
"multi-label gold, model missed one",64,0.29
missed only,59,0.27
very short (under 12 words),15,0.07
gold-empty (no label survived the vote),12,0.05
truncated (over 256 tokens),2,0.01


In [4]:
for _, r in errs.sample(12, random_state=1).iterrows():
    print("-" * 80)
    print("TEXT:     ", r["text"][:400])
    print("TRUE:     ", r["true"] or "(none)")
    print("PREDICTED:", r["predicted"] or "(none)")
    print("TAGS:     ", r["tags"])

--------------------------------------------------------------------------------
TEXT:      Our Invite-a-Friend feature is located in various areas of certain Barnes & Noble Websites and within certain Devices. By clicking on 'Email' or 'Find friends from contacts' on your Device, customers can email a link to a friend which will invite the friend to view the Barnes & Noble Website page or attend the event described on that page.
TRUE:      Other
PREDICTED: First Party Collection/Use
TAGS:      ['Other boundary', 'swap (missed one, added another)']
--------------------------------------------------------------------------------
TEXT:      Other Information We Collect Online Other Information is any information other than Personal Information that does not reveal your specific identity or does not directly relate to an individual, such as browser information, information collected through cookies, pixel tags and other technologies, demographic information, other information provided by 